# Chest X-Ray Disease Classification — Simple CNN
**Kaggle Cup 2026 | ML Wing Axios**

A simple CNN built with Keras + TensorFlow for 12-class chest disease classification.

Step 1: Import libraries & paths
Step 2: resolve path from subfolders and create a new column in df for that full_path
Step 3: Label encode those labels in df
Step 4: train test split
Step 5: 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

In [ ]:
BASE_DIR="/kaggle/input/competitions/ml-wing-axios-kaggle-cup-2026/Kaggle Cup 2026 Dataset"
train_path="/kaggle/input/competitions/ml-wing-axios-kaggle-cup-2026/Kaggle Cup 2026 Dataset/train"
test_path="/kaggle/input/competitions/ml-wing-axios-kaggle-cup-2026/Kaggle Cup 2026 Dataset/test"
labels_path ="/kaggle/input/competitions/ml-wing-axios-kaggle-cup-2026/Kaggle Cup 2026 Dataset/train_labels.csv"
df=pd.read_csv(labels_path)
img_size=192
batch_size=32
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
def resolve_path(fp):
    if os.path.isabs(fp) and os.path.exists(fp):
        return fp
    candidate = os.path.join(BASE_DIR, fp)
    if os.path.exists(candidate):
        return candidate
    
    for root,dir,files in os.walk(os.path.join(BASE_DIR, "train")):
        file_name=os.path.basename(fp)
        if file_name in files:
            return os.path.join(root,file_name)

df['full_path'] = df['FilePath'].apply(resolve_path)

le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df['Label'])

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['encoded_label'], random_state=42
)

In [ ]:
def load_image(path, label):
    img=tf.io.read_file(path)
    img=tf.image.decode_png(img, channels=3) 
    img=tf.image.resize(img, [img_size, img_size])
    img=tf.cast(img, tf.float32)/255.0
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    return img, label

def make_dataset(dataframe, augment_data=False, shuffle=False):
    paths  = dataframe['full_path'].values
    labels = dataframe['encoded_label'].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, augment_data=True,  shuffle=True)
val_ds   = make_dataset(val_df,   augment_data=False, shuffle=False)

In [ ]:
def build_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.35)(x)

    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs)



model = build_cnn((img_size, img_size, 3), 12)

In [ ]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        '/kaggle/working/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-6
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True
    )
]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks
)

In [ ]:
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

In [ ]:
test_records = []
for fname in sorted(os.listdir(test_path)):
    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        image_id  = os.path.splitext(fname)[0]   
        full_path = os.path.join(test_path, fname)
        test_records.append({'ImageId': image_id, 'full_path': full_path})

test_df = pd.DataFrame(test_records)

In [ ]:
def load_test_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_size, img_size])
    img = tf.cast(img, tf.float32) / 255.0
    return img

test_ds = (
    tf.data.Dataset
    .from_tensor_slices(test_df['full_path'].values)
    .map(load_test_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

raw_preds    = model.predict(test_ds, verbose=1)
pred_indices = np.argmax(raw_preds, axis=1)
pred_labels  = le.inverse_transform(pred_indices)

In [ ]:
submission = pd.DataFrame({
    'ImageId': test_df['ImageId'].values,
    'Label':   pred_labels
})

submit_path = '/kaggle/working/submission.csv'
submission.to_csv(submit_path, index=False)